In [1]:
import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import ocha_stratus as stratus


In [2]:
IBTRACS_PATH = "C:/Users/pauni/Downloads/ibtracs.since1980.list.v04r01.csv"

ds = xr.open_dataset(IBTRACS_PATH)


ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy', 'cfgrib', 'rasterio']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
moz_adm1 = stratus.codab.load_codab_from_fieldmaps("MOZ", admin_level=1)
mdg_adm0 = stratus.codab.load_codab_from_fieldmaps("MDG", admin_level=0)
bgd_adm2 = stratus.codab.load_codab_from_fieldmaps("BGD", admin_level=2)


In [ ]:
df = ds[
    [
        "sid",        # storm ID
        "season",     # year
        "lat",
        "lon",
        "usa_wind",   # 1-min sustained wind (kt)
    ]
].to_dataframe().reset_index()

df = df.rename(
    columns={
        "usa_wind": "wind_kt",
        "lat": "latitude",
        "lon": "longitude",
    }
)

# basic cleaning
df = df.dropna(subset=["wind_kt", "latitude", "longitude"])
df = df[df["season"] >= 1980]


In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326",
)


In [ ]:
gdf_64kt = gdf[gdf["wind_kt"] >= 64]


In [ ]:
storm_count = (
    gdf_64kt[["sid", "season"]]
    .drop_duplicates()
    .groupby("season")
    .size()
)

print(storm_count)
